In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [2]:
# Load data
users = pd.read_csv("users.csv")
products = pd.read_csv("products.csv")
orders = pd.read_csv("orders.csv")
order_products = pd.read_csv("order_products.csv")

# Merge everything
data = order_products.merge(orders, on="order_id") \
                     .merge(products, on="product_id")


In [3]:
# User-level feature
user_order_counts = orders.groupby("user_id")["order_id"].count()
data["user_total_orders"] = data["user_id"].map(user_order_counts)

# Product-level features
product_purchase_counts = data.groupby("product_id").size()
product_reorder_rates = data.groupby("product_id")["reordered"].mean()

data["product_total_purchases"] = data["product_id"].map(product_purchase_counts)
data["product_reorder_rate"] = data["product_id"].map(product_reorder_rates)

# User–product interaction feature
user_product_counts = (
    data.groupby(["user_id", "product_id"])
        .size()
        .rename("user_product_purchase_count")
        .reset_index()
)

data = data.merge(
    user_product_counts,
    on=["user_id", "product_id"],
    how="left"
)


In [4]:
features = [
    "user_total_orders",
    "product_total_purchases",
    "product_reorder_rate",
    "user_product_purchase_count",
    "category"
]

X = data[features]
y = data["reordered"]


In [5]:
numeric_features = [
    "user_total_orders",
    "product_total_purchases",
    "product_reorder_rate",
    "user_product_purchase_count"
]

categorical_features = ["category"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)

print("Model trained successfully.")


Model trained successfully.


In [7]:
def recommend_products_for_user(user_id, top_n=5):
    user_data = products.copy()

    user_data["user_id"] = user_id

    user_total_orders = orders[orders["user_id"] == user_id].shape[0]
    user_data["user_total_orders"] = user_total_orders

    user_product_counts = (
        data[data["user_id"] == user_id]
        .groupby("product_id")
        .size()
        .to_dict()
    )

    user_data["user_product_purchase_count"] = user_data["product_id"].map(
        user_product_counts
    ).fillna(0)

    user_data["product_total_purchases"] = user_data["product_id"].map(
        product_purchase_counts
    )

    user_data["product_reorder_rate"] = user_data["product_id"].map(
        product_reorder_rates
    )

    X_user = user_data[features]

    user_data["reorder_probability"] = model.predict_proba(X_user)[:, 1]

    recommendations = user_data.sort_values(
        "reorder_probability", ascending=False
    ).head(top_n)

    return recommendations[
        ["product_id", "category", "reorder_probability"]
    ]


In [12]:
print(recommend_products_for_user(user_id=60, top_n=5))


    product_id    category  reorder_probability
28          29      frozen             0.957012
63          64       dairy             0.932690
40          41  vegetables             0.857090
26          27      drinks             0.788623
72          73       dairy             0.760778


In [11]:
import joblib

joblib.dump(model, "reorder_model.joblib")
joblib.dump(product_purchase_counts, "product_purchase_counts.joblib")
joblib.dump(product_reorder_rates, "product_reorder_rates.joblib")

products.to_csv("products.csv", index=False)
orders.to_csv("orders.csv", index=False)

print("Model and artefacts exported successfully.")


Model and artefacts exported successfully.


In [13]:
user_product_counts = (
    data.groupby(["user_id", "product_id"])
        .size()
        .rename("user_product_purchase_count")
        .reset_index()
)

user_product_counts.to_csv(
    "user_product_purchase_counts.csv",
    index=False
)
